# Observables and expectation values
**Quantum Computing · Week 4 (Entanglement) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qc/w4d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Measuring $Z$ gives $+1$ (for 0) or $-1$ (for 1). Its average over many shots is the
**expectation value** $\langle Z\rangle = P(0) - P(1)$: one smooth number that summarizes the state.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

qc = QuantumCircuit(1)
qc.ry(1.0, 0)                       # tilt |0> by 1 radian
sv = Statevector(qc)
p = sv.probabilities()

print("P(0), P(1)            :", p.round(3))
print("<Z> = P(0) - P(1)     :", round(p[0] - p[1], 3))
print("<Z> computed by Qiskit:", round(float(np.real(sv.expectation_value(SparsePauliOp("Z")))), 3))
print("cos(theta)            :", round(np.cos(1.0), 3), "  <- the z-coordinate on the Bloch sphere")

## The Estimator primitive
`StatevectorEstimator` computes expectation values for many parameter values at once.
This is the call a variational quantum model makes over and over.

In [ ]:
import matplotlib.pyplot as plt
from qiskit.circuit import Parameter
from qiskit.primitives import StatevectorEstimator

theta = Parameter("θ")
ansatz = QuantumCircuit(1)
ansatz.ry(theta, 0)

angles = np.linspace(0, 2 * np.pi, 41).reshape(-1, 1)        # one row per parameter setting
estimator = StatevectorEstimator()
result = estimator.run([(ansatz, SparsePauliOp("Z"), angles),
                        (ansatz, SparsePauliOp("X"), angles)]).result()

plt.plot(angles, result[0].data.evs, "o-", ms=4, label=r"$\langle Z\rangle$")
plt.plot(angles, result[1].data.evs, "o-", ms=4, label=r"$\langle X\rangle$")
plt.xlabel("θ (radians)")
plt.ylabel("expectation value")
plt.title("RY(θ): the arrow's z and x coordinates")
plt.legend()
plt.show()

## Estimating from shots
Real hardware only gives shots, so $\langle Z\rangle$ is estimated as (number of 0s − number of 1s) / shots.

In [ ]:
from qiskit.primitives import StatevectorSampler

measured = qc.copy()
measured.measure_all()
sampler = StatevectorSampler(seed=33)
for shots in [10, 100, 1000, 10000]:
    counts = sampler.run([measured], shots=shots).result()[0].data.meas.get_counts()
    estimate = (counts.get("0", 0) - counts.get("1", 0)) / shots
    print(f"{shots:>6} shots:  <Z> ≈ {estimate:+.3f}   (exact {np.cos(1.0):+.3f})")

## Measuring X: rotate first
Hardware measures in the 0/1 (Z) basis. To measure $X$, apply H first: it swaps the x- and z-axes.

In [ ]:
plus = QuantumCircuit(1)
plus.h(0)                                 # prepare |+>, which points along +x

in_z = plus.copy()
in_z.measure_all()
in_x = plus.copy()
in_x.h(0)                                 # rotate x onto z, then measure
in_x.measure_all()

print("|+> measured in the Z basis:", sampler.run([in_z], shots=1000).result()[0].data.meas.get_counts())
print("|+> measured in the X basis:", sampler.run([in_x], shots=1000).result()[0].data.meas.get_counts())

## Two-qubit observables
`ZZ` is $+1$ when the two bits agree and $-1$ when they differ, so it measures correlation.
(Pauli labels are read like bitstrings: the rightmost letter acts on qubit 0.)

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell_state = Statevector(bell)
for label in ["IZ", "ZI", "ZZ", "XX"]:
    value = np.real(bell_state.expectation_value(SparsePauliOp(label)))
    print(f"<{label}> = {value:+.3f}")

Each qubit alone averages to 0 (a fair coin), but $\langle ZZ\rangle = +1$: they always agree.

## Your turn: the "energy" of a state
Observables can be added: $\langle aA + bB\rangle = a\langle A\rangle + b\langle B\rangle$. Take
$H = ZZ + 0.5\,XX$. Compute $\langle H\rangle$ for $|00\rangle$, $|\Phi^+\rangle$ and $|\Psi^-\rangle$, first from the
Bell-state sign table (yesterday's lesson), then with Qiskit. Which state has the **lowest** value?

In [ ]:
H = SparsePauliOp.from_list([("ZZ", 1.0), ("XX", 0.5)])
s = 1 / np.sqrt(2)
states = {"|00>": [1, 0, 0, 0], "Phi+": [s, 0, 0, s], "Psi-": [0, s, -s, 0]}
my_guess = {"|00>": None, "Phi+": None, "Psi-": None}      # TODO: your predictions from <ZZ> and <XX>
for name, v in states.items():
    value = None                                           # TODO: Statevector(v).expectation_value(H).real
    print(f"{name:>5}: predicted {my_guess[name]}, Qiskit says {value}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
my_guess = {"|00>": 1 + 0.5 * 0, "Phi+": 1 + 0.5 * 1, "Psi-": -1 + 0.5 * (-1)}
for name, v in states.items():
    value = Statevector(v).expectation_value(H).real
    print(f"{name:>5}: predicted {my_guess[name]:+.2f}, Qiskit says {value:+.2f}")
print("Psi- has the lowest value: it is the 'ground state' of H. Finding the lowest-energy state of an observable")
print("is exactly what the variational quantum eigensolver (week 7) does, one expectation value at a time.")